Preparacion 

In [12]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

DATOS = Path("../data/PlantVillage")

una función que recibe la ruta de una foto y devuelve sus características:

In [13]:
def extraer_caracteristicas(ruta):
    matriz= np.array(Image.open(ruta).convert("RGB"))
    promedio = matriz.mean(axis=(0,1))#cuanto vale en promedio cada canal de color
    desvio = matriz.std(axis=(0,1)) #cuanto varia cada canal de color 
    return {
        "promedio_rojo": promedio[0],
        "promedio_verde": promedio[1],
        "promedio_azul": promedio[2],
        "desvio_rojo": desvio[0],
        "desvio_verde": desvio[1],
        "desvio_azul": desvio[2]

    }

def crea una función: un bloque de código con nombre que podés usar muchas veces.
axis=(0, 1) significa “promediá a lo alto y a lo ancho”, así queda un solo número por canal de color.
El desvío mide cuánto cambian los valores dentro de la foto. Una hoja lisa y pareja tiene desvío bajo; una con manchas, alto.

Tiene que dar rojo_prom cerca de 97.6 y verde_prom cerca de 109.0, con el verde como el más alto.

In [14]:
carpeta= DATOS / "Tomato_Late_blight"
prueba= list(carpeta.iterdir())[0]
extraer_caracteristicas(prueba)

{'promedio_rojo': np.float64(125.44126892089844),
 'promedio_verde': np.float64(117.66810607910156),
 'promedio_azul': np.float64(101.712890625),
 'desvio_rojo': np.float64(63.149606264542484),
 'desvio_verde': np.float64(54.541259543078674),
 'desvio_azul': np.float64(59.554949062653264)}

ruta.suffix es la extensión del archivo, por ejemplo .JPG.
.lower() la pasa a minúsculas, así .JPG y .jpg cuentan como lo mismo.
in EXTENSIONES pregunta si está en la lista de válidas. El archivo basura no tiene extensión, así que queda afuera.

In [15]:
EXTENSIONES= [".jpg", ".jpeg", ".png"]
filas=[]
for carpeta in DATOS.iterdir():
    if carpeta.name.startswith("Tomato"):
        for ruta in carpeta.iterdir():
            if ruta.suffix.lower() in EXTENSIONES:
                fila=extraer_caracteristicas(ruta)
                fila["clase"]= carpeta.name
                filas.append(fila)
tabla= pd.DataFrame(filas)
tabla.shape            

(16011, 7)

Ya van tres sorpresas en estos datos: la carpeta duplicada, el desbalance y ahora un archivo basura. Es lo normal. Se suele decir que un científico de datos pasa la mayor parte del tiempo limpiando datos y no entrenando modelos, y lo estás comprobando en tu propio proyecto.

Pregunta: ¿cuántas filas hay de cada clase?

tabla["clase"] selecciona la columna clase.
.value_counts() cuenta cuántas veces aparece cada valor.
Sirve como control: cada fila es una foto, así que el resultado tiene que coincidir con la cantidad de fotos de cada carpeta. Si apareciera una sola clase, o números raros, sabríamos que la tabla se armó mal, como pasó con el error de carpetas.

In [16]:
tabla["clase"].value_counts()

clase
Tomato__Tomato_YellowLeaf__Curl_Virus          3208
Tomato_Bacterial_spot                          2127
Tomato_Late_blight                             1909
Tomato_Septoria_leaf_spot                      1771
Tomato_Spider_mites_Two_spotted_spider_mite    1676
Tomato_healthy                                 1591
Tomato__Target_Spot                            1404
Tomato_Early_blight                            1000
Tomato_Leaf_Mold                                952
Tomato__Tomato_mosaic_virus                     373
Name: count, dtype: int64

tabla.groupby("clase")["promedio_verde"].mean().sort_values()
Pregunta: ¿cuánto verde tienen, en promedio, las fotos de cada clase?

el resultado sería enferma 70 y sana 115: un número por clase.

In [17]:
tabla.groupby("clase")["promedio_verde"].mean().sort_values()

clase
Tomato_Bacterial_spot                          108.668254
Tomato_Early_blight                            113.035147
Tomato_Late_blight                             115.166323
Tomato__Tomato_mosaic_virus                    116.636025
Tomato_Leaf_Mold                               117.227910
Tomato_healthy                                 118.613556
Tomato_Spider_mites_Two_spotted_spider_mite    119.651030
Tomato__Target_Spot                            120.036321
Tomato__Tomato_YellowLeaf__Curl_Virus          120.171928
Tomato_Septoria_leaf_spot                      133.070353
Name: promedio_verde, dtype: float64

Paso 7: entrenar tu primer modelo

Antes del código, la idea más importante de todo machine learning: nunca se evalúa un modelo con los mismos datos con los que aprendió. Sería como tomar un examen con las mismas preguntas que se usaron para estudiar: no sabés si el alumno entendió o memorizó. Por eso se divide la tabla en dos:

Entrenamiento (80%): el modelo aprende de estas fotos.
Prueba (20%): fotos que el modelo nunca vio y que usamos para ponerle nota.

separar features y target:

In [18]:
x=tabla.drop(columns="clase")# las 6 columnas de números
y=tabla["clase"]# lo que queremos predecir

dividir en entrenamiento y prueba

test_size=0.2 reserva el 20% para la prueba.
random_state=42 fija el sorteo, para que el resultado sea el mismo cada vez que lo ejecutes.
stratify=y mantiene la proporción de cada clase en las dos partes, importante por el desbalance que encontraste.
Tiene que mostrar 12808 3203.



In [21]:
from sklearn.model_selection import train_test_split
x_entreno, x_prueba, y_entreno, y_prueba = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(len(x_entreno),len(x_prueba))

12808 3203


entrenar y evaluar

Reconocés el patrón de la clase 1: fit para aprender y predict para usar lo aprendido. El modelo es un Random Forest (bosque aleatorio): cientos de árboles de decisión que votan entre todos. Más adelante te explico cómo funciona por dentro.

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

modelo= RandomForestClassifier(random_state=42)
modelo.fit(x_entreno, y_entreno) #aprendemos de los datos de entrenamiento

predicciones=modelo.predict(x_prueba) #predecimos con los datos de prueba que nunca vio
accuracy_score(y_prueba,predicciones) #porcentajes de aciertos 

0.7165157664689353

Qué significa exactamente 0.72
Ese número se llama exactitud (accuracy) y es la proporción de aciertos sobre el total de fotos de prueba:

0.72 equivale a 72%.
El conjunto de prueba tenía 3.203 fotos que el modelo nunca había visto.
Acertó la enfermedad en unas 2.300 y se equivocó en unas 900.
La escala va de 0 (no acierta ninguna) a 1 (acierta todas). Una respuesta completa sería: “el modelo clasifica bien 72 de cada 100 hojas que nunca vio”.

In [23]:
from sklearn.metrics import classification_report

print(classification_report(y_prueba, predicciones))

                                             precision    recall  f1-score   support

                      Tomato_Bacterial_spot       0.76      0.79      0.78       426
                        Tomato_Early_blight       0.49      0.34      0.40       200
                         Tomato_Late_blight       0.69      0.63      0.66       382
                           Tomato_Leaf_Mold       0.76      0.59      0.67       190
                  Tomato_Septoria_leaf_spot       0.77      0.74      0.75       354
Tomato_Spider_mites_Two_spotted_spider_mite       0.62      0.67      0.64       335
                        Tomato__Target_Spot       0.58      0.64      0.61       281
      Tomato__Tomato_YellowLeaf__Curl_Virus       0.77      0.85      0.81       642
                Tomato__Tomato_mosaic_virus       0.73      0.53      0.62        75
                             Tomato_healthy       0.82      0.89      0.86       318

                                   accuracy                    